In [ ]:
# Phase 7 slice-window averaging. No training: the c2 and c3 confirm checkpoints
# re-read their own held-out folds from prep v2 artifacts stored at 9 slice groups,
# once at the 5 groups they trained on and once at all 9.
import glob, os, shutil, sys, time

GIT_SHA = '801df73-wip'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

# A stale rsna-knee-src version cost 7.5 minutes of a Phase 5 screen before the
# run noticed it was training the wrong code (NOTES 2026-09-05). These asserts
# fail in the first cell instead.
from knee.dataset import PreppedSlotDataset
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# A stale src dataset has now cost three runs. The old guard listed the previous
# change by hand, so it only ever caught yesterday's mistake -- this one is
# derived from what the configs below actually ask for, so a config needing new
# code cannot silently outrun the dataset version. Tuples, not sets: this cell is
# an f-string and braces here would be interpolated.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel.forward', KneeModel.forward, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('differential_param_groups', differential_param_groups, ('backbone_lr', 'head_lr')),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__, ('weights_df',)),
    ('masked_bce_loss', masked_bce_loss, ('weights',)),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, 'attention_weights'), 'rsna-knee-src predates the c2 head'
print('src carries every Phase 6 change this run depends on | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# Each checkpoint predicts its held-out fold and gold at both reads. The model
# is loaded once per fold and both reads run on it, so the pair differs only in
# which stored groups the loader hands over.
MEMBERS = [
    # confirm name, backbone, head, slots it reads
    ('c2_6slot_attn', 'efficientnet_b0', 'slot_attention', slot_names),
    ('c3_dinov2_2slot', 'vit_small_patch14_dinov2.lvd142m', 'mean', slot_names[:2]),
]
READS = (5, 9)
OUT_SIZE, BATCH = 224, 8

def mounted_one(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    assert len(hits) == 1, f'{pattern}: expected exactly one mounted file, found {hits}'
    return hits[0]

def loader(uids, labels_df, cfg_slots, n_groups):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=n_groups,
                            out_size=OUT_SIZE, slots=cfg_slots)
    return DataLoader(ds, batch_size=BATCH, num_workers=2)

eval_labels = LABEL_SOURCES['steven_v4'][1]
labeled_uids = [u for u in all_uids if u not in holdout and u in uid_to_npz]
row_of = {u: i for i, u in enumerate(labeled_uids)}

oof_true = None
confirm_oof, confirm_gold = {}, {}
preds, gold_preds = {}, {}
reproduced, moved_by = {}, {}
t_start = time.time()
for name, backbone, head, cfg_slots in MEMBERS:
    saved_uids = pd.read_csv(mounted_one(f'oof_uids_{name}.csv'))['StudyInstanceUID'].astype(str)
    assert saved_uids.tolist() == labeled_uids, f'{name}: saved OOF rows are in a different order'
    member_true = np.load(mounted_one(f'oof_true_{name}.npy'))
    assert oof_true is None or np.array_equal(np.nan_to_num(oof_true, nan=-1),
                                              np.nan_to_num(member_true, nan=-1))
    oof_true = member_true
    confirm_oof[name] = np.load(mounted_one(f'oof_pred_{name}.npy'))
    confirm_gold[name] = np.load(mounted_one(f'gold_folds_{name}.npy')).mean(axis=0)

    ckpts = sorted(glob.glob(f'/kaggle/input/**/{name}_fold[0-4].pt', recursive=True))
    assert [c[-8:] for c in ckpts] == [f'fold{k}.pt' for k in range(5)], f'{name}: {ckpts}'
    for n_groups in READS:
        preds[(name, n_groups)] = np.full_like(oof_true, np.nan)
    gold_by_read = {n_groups: [] for n_groups in READS}

    for fold, ckpt in enumerate(ckpts):
        extra = {'img_size': OUT_SIZE} if backbone.startswith('vit_') else {}
        model = KneeModel(backbone_name=backbone, num_labels=12, pretrained=False,
                          head=head, **extra)
        model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=True))
        model.to(device).eval()
        _, va_uids = train_val_split(folds, val_fold=fold, exclude_uids=holdout)
        va_uids = [u for u in va_uids if u in uid_to_npz]
        rows = [row_of[u] for u in va_uids]
        for n_groups in READS:
            _, y_pred = evaluate(model, loader(va_uids, eval_labels, cfg_slots, n_groups),
                                 device=device)
            preds[(name, n_groups)][rows] = y_pred
            g_true, g_pred = evaluate(model, loader(gold_uids, gold_labels, cfg_slots, n_groups),
                                      device=device)
            gold_by_read[n_groups].append(g_pred)
        print(f'{name} fold {fold}: {len(va_uids)} val studies at {READS} groups '
              f'({(time.time() - t_start) / 60:.1f} min elapsed)')
        del model
        torch.cuda.empty_cache()

    for n_groups in READS:
        assert not np.isnan(preds[(name, n_groups)]).any(), f'{name}: a study was never predicted'
        gold_preds[(name, n_groups)] = np.mean(gold_by_read[n_groups], axis=0)

    # The reproduction gate. The 5-group read sees the confirm's training positions
    # through the same model, so its predictions should match the confirm's up to
    # floating point and batch composition. The prep9 kernels compare pixels
    # against the phase6 artifacts; a JPEG encoder change between Kaggle images
    # could still move a prediction by ~1e-3, which is why the assert is on the
    # macro rather than on every entry.
    worst = float(np.abs(preds[(name, 5)] - confirm_oof[name]).max())
    macro5 = macro_auc(oof_true, preds[(name, 5)])
    macro_saved = macro_auc(oof_true, confirm_oof[name])
    print(f'{name}: 5-group read vs confirm -- macro {macro5:.5f} vs {macro_saved:.5f}, '
          f'max |prediction diff| {worst:.2e}')
    reproduced[name] = abs(macro5 - macro_saved) < 5e-4

    # A 9-group read that silently handed over the 5-group tensors would report
    # "no gain" instead of failing. The local dry run cannot catch that -- every
    # stored group in the local sample is the same slice -- so it is checked on
    # real stacks.
    moved_by[name] = float(np.abs(preds[(name, 9)] - preds[(name, 5)]).max())
    print(f'{name}: max |prediction change| from 5 to 9 groups {moved_by[name]:.2e}')

gold_true = g_true
for name, n_groups in preds:
    np.save(f'/kaggle/working/oof_pred_{name}_g{n_groups}.npy', preds[(name, n_groups)])
    np.save(f'/kaggle/working/gold_pred_{name}_g{n_groups}.npy', gold_preds[(name, n_groups)])
np.save('/kaggle/working/oof_true.npy', oof_true)
np.save('/kaggle/working/gold_true.npy', gold_true)
pd.DataFrame({'StudyInstanceUID': labeled_uids}).to_csv('/kaggle/working/oof_uids.csv', index=False)

# Asserted only now, after every array is on disk and across both members, so a
# failed gate still leaves the predictions behind to diagnose it with.
not_reproduced = [name for name, ok in reproduced.items() if not ok]
assert not not_reproduced, f'the 5-group read does not reproduce the confirm for {not_reproduced}'
assert max(moved_by.values()) > 1e-4, (
    'no member changed a prediction between reads -- is the 9-group read reading 9 groups?')
unmoved = [name for name, moved in moved_by.items() if moved <= 1e-4]
if unmoved:
    print(f'NOTE: {unmoved} changed no prediction between reads; the other member did')

In [ ]:
# Pooled OOF (n~4,349) decides; gold (n=58) is reported beside it. Adopt 9 groups
# in the next submission only where the OOF CI excludes zero.
def report(label, p5, p9, g5, g9):
    d, lo, hi = paired_macro_auc_delta(oof_true, p5, p9)
    gd, glo, ghi = paired_macro_auc_delta(gold_true, g5, g9)
    verdict = 'CI excludes zero' if lo > 0 or hi < 0 else 'CI spans zero'
    print(f'{label:24s} OOF {macro_auc(oof_true, p5):.4f} -> {macro_auc(oof_true, p9):.4f}  '
          f'{d:+.4f} [{lo:+.4f}, {hi:+.4f}] {verdict}')
    print(f'{"":24s} gold {macro_auc(gold_true, g5):.4f} -> {macro_auc(gold_true, g9):.4f}  '
          f'{gd:+.4f} [{glo:+.4f}, {ghi:+.4f}]')

names = [m[0] for m in MEMBERS]
for name in names:
    report(name, preds[(name, 5)], preds[(name, 9)], gold_preds[(name, 5)], gold_preds[(name, 9)])

def ensemble(n_groups, store):
    return np.mean([store[(name, n_groups)] for name in names], axis=0)

# the Submission A artifact at 5 groups; its pooled OOF was recorded as 0.8641
report('c2+c3 probability mean', ensemble(5, preds), ensemble(9, preds),
       ensemble(5, gold_preds), ensemble(9, gold_preds))
print(f'\ntotal {(time.time() - t_start) / 60:.1f} min')